# HW2 Part 1 — Data Cleaning and Validation

Goal: clean the dirty `ohlc_hw` table and save a validated copy as
`ohlc_hw_cleaned` in `data/data.db`. The raw table is never modified.

## 1.1 Load and inspect the data

In [1]:
import pandas as pd
from pathlib import Path
import sqlite3

db_path = Path("data/data.db").resolve()

if not db_path.is_file():
    raise FileNotFoundError(f"Database not found: {db_path}")

# Open the database in read-only mode for initial exploration.
with sqlite3.connect(db_path.as_uri() + "?mode=ro", uri=True) as connection:
    df = pd.read_sql_query("SELECT * FROM ohlc_hw", connection)

print("Number of rows and columns:", df.shape)
df.head()

Number of rows and columns: (397711, 8)


,ts,open,high,low,close,adj_close,volume,ticker
0,1975-04-14,5.4300,5.4800,5.3600,5.4400,1.640000,446392,CAT
1,1978-04-05,4.8400,4.8900,4.8100,4.8800,0.656990,355200,CVX
2,2018-05-25,124.5400,124.7600,121.2200,122.1900,92.220000,8936649,CVX
3,1995-12-07,9.9700,9.9700,9.8300,9.9400,5.880000,3954150,HD
4,1981-07-23,0.1038,0.1038,0.1038,0.1038,0.080013,34451232,AAPL


In [2]:
# Inspect column types and non-missing values.
df.info()

# Count missing values in each column.
display(df.isna().sum().rename("missing_count").to_frame())

# Summarize numerical columns.
display(df.describe())

<class 'pandas.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   ts         397711 non-null  str    
 1   open       397711 non-null  float64
 2   high       397711 non-null  float64
 3   low        397711 non-null  float64
 4   close      397711 non-null  float64
 5   adj_close  377856 non-null  float64
 6   volume     397711 non-null  int64  
 7   ticker     397711 non-null  str    
dtypes: float64(5), int64(1), str(2)
memory usage: 24.3 MB


,missing_count
ts,0
open,0
high,0
low,0
close,0
adj_close,19855
volume,0
ticker,0


,open,high,low,close,adj_close,volume
count,397711.000000,397711.000000,397711.000000,397711.000000,377856.000000,3.977110e+05
mean,46.285479,46.747660,45.816279,46.292903,51.652250,3.097773e+07
std,67.132594,67.778210,66.477178,67.141879,194.584045,1.266237e+08
min,0.000000,0.005208,0.004801,0.005208,0.002945,0.000000e+00
25%,4.209938,4.240000,4.160000,4.200000,1.390000,1.730700e+06
50%,19.850000,20.060000,19.630000,19.860000,11.960000,4.648200e+06
75%,57.530000,58.130000,56.950000,57.530000,43.770000,1.102380e+07
max,604.260000,608.630000,597.880000,605.400000,10442.510783,9.276606e+09


**Columns**

| Column | Meaning |
|---|---|
| `ts` | trading date, stored as text `YYYY-MM-DD` |
| `ticker` | stock symbol (some labels contain HTML markup) |
| `open`, `high`, `low`, `close` | raw daily prices in USD |
| `adj_close` | vendor close adjusted for splits and dividends |
| `volume` | number of shares traded that day |

First observations: `adj_close` is the only column with missing values,
and the minimum `open` is 0, which is impossible for a listed stock.

In [3]:
# Inspect original ticker labels before normalization.
print("Number of distinct original ticker labels:", df["ticker"].nunique())
print("Original ticker labels:")
print(sorted(df["ticker"].unique().tolist()))

# Parse dates without changing the original data.
parsed_dates = pd.to_datetime(df["ts"], errors="coerce")

print("Invalid or missing dates:", parsed_dates.isna().sum())
print("Earliest date:", parsed_dates.min())
print("Latest date:", parsed_dates.max())

# Count fully identical rows.
print("Exact duplicate rows:", df.duplicated().sum())

Number of distinct original ticker labels: 150
Original ticker labels:
['<a target="_blank"><span style="color: blue">AAPL</span></a>', '<a target="_blank"><span style="color: blue">AMGN</span></a>', '<a target="_blank"><span style="color: blue">AMZN</span></a>', '<a target="_blank"><span style="color: blue">AXP</span></a>', '<a target="_blank"><span style="color: blue">BA</span></a>', '<a target="_blank"><span style="color: blue">CAT</span></a>', '<a target="_blank"><span style="color: blue">CRM</span></a>', '<a target="_blank"><span style="color: blue">CSCO</span></a>', '<a target="_blank"><span style="color: blue">CVX</span></a>', '<a target="_blank"><span style="color: blue">DIS</span></a>', '<a target="_blank"><span style="color: blue">GS</span></a>', '<a target="_blank"><span style="color: blue">HD</span></a>', '<a target="_blank"><span style="color: blue">HON</span></a>', '<a target="_blank"><span style="color: blue">IBM</span></a>', '<a target="_blank"><span style="color: blue"

In [4]:
import re
from html import unescape

# Keep the original dataset unchanged.
working = df.copy()

# Preserve the original ticker labels for auditing.
working["ticker_original"] = working["ticker"]

def normalize_ticker(value):
    """Remove HTML markup and whitespace, then uppercase the ticker."""
    if pd.isna(value):
        return pd.NA
    text = unescape(str(value))
    text = re.sub(r"<[^>]*>", "", text)
    return text.strip().upper()

working["ticker"] = working["ticker"].map(normalize_ticker)
working["ts"] = pd.to_datetime(working["ts"], errors="coerce")

# Display the mapping between original and normalized labels.
ticker_mapping = (
    working[["ticker_original", "ticker"]]
    .drop_duplicates()
    .sort_values(["ticker", "ticker_original"])
)

display(ticker_mapping.head(20))
print("Number of normalized tickers:", working["ticker"].nunique())
print("Missing ticker labels:", working["ticker"].isna().sum())
print("Empty ticker labels:", working["ticker"].eq("").sum())

,ticker_original,ticker
102,"<a target=""_blank""><span style=""color: blue"">A...",AAPL
1121,"<a target=""_blank""><span style=""color: red"">AA...",AAPL
825,"<a target=""_blank""><span style=""font-weight: b...",AAPL
744,"<a target=""_blank""><span style=""font-weight: b...",AAPL
4,AAPL,AAPL
5005,"<a target=""_blank""><span style=""color: blue"">A...",AMGN
45,"<a target=""_blank""><span style=""color: red"">AM...",AMGN
368,"<a target=""_blank""><span style=""font-weight: b...",AMGN
1227,"<a target=""_blank""><span style=""font-weight: b...",AMGN
33,AMGN,AMGN


Number of normalized tickers: 30
Missing ticker labels: 0
Empty ticker labels: 0


In [5]:
# Date range and number of rows for each normalized ticker.
ticker_overview = (
    working.groupby("ticker")["ts"]
    .agg(first_date="min", last_date="max", rows="size")
)

print("Tickers:", ", ".join(ticker_overview.index))
display(ticker_overview)

Tickers: AAPL, AMGN, AMZN, AXP, BA, CAT, CRM, CSCO, CVX, DIS, GS, HD, HON, IBM, JNJ, JPM, KO, MCD, MMM, MRK, MSFT, NKE, NVDA, PG, SHW, TRV, UNH, V, VZ, WMT


,first_date,last_date,rows
ticker,,,
AAPL,1980-12-12,2024-10-18,11942
AMGN,1983-06-17,2024-10-18,11290
AMZN,1997-05-15,2024-12-06,7458
AXP,1972-04-03,2024-10-18,14260
BA,1962-01-02,2024-10-18,16980
CAT,1962-01-02,2024-10-18,17044
CRM,2004-06-23,2024-10-18,5525
CSCO,1990-02-16,2024-10-18,9427
CVX,1962-01-02,2024-12-06,17073


### Repeated records

Duplicates are checked *after* ticker normalization, otherwise
`<span>AAPL</span>` and `AAPL` would look like two different rows.
Exact copies are removed; rows sharing a (ticker, ts) key with different
values would need investigation, so they are counted separately.

In [6]:
# Compare the eight data columns, excluding the original label metadata.
data_columns = df.columns.tolist()

duplicate_mask = working.duplicated(subset=data_columns)
exact_duplicates = working.loc[duplicate_mask].copy()

# Remove only observations with identical normalized data.
deduplicated = working.loc[~duplicate_mask].copy()
deduplicated = deduplicated.sort_values(["ticker", "ts"]).reset_index(drop=True)

# Identify remaining observations sharing a ticker and date.
conflict_mask = deduplicated.duplicated(
    subset=["ticker", "ts"], keep=False
)
conflicting_rows = deduplicated.loc[conflict_mask].copy()

print("Identical observations removed:", len(exact_duplicates))
print("Remaining observations:", len(deduplicated))
print("Rows with conflicting ticker-date keys:", len(conflicting_rows))

display(conflicting_rows.head(20))

Identical observations removed: 29460
Remaining observations: 368251
Rows with conflicting ticker-date keys: 0


,ts,open,high,low,close,adj_close,volume,ticker,ticker_original


## 1.2 Detect data quality issues

Each row gets boolean flags; nothing is changed at this stage. One row can
have several issues at once.

In [7]:
import numpy as np

price_columns = ["open", "high", "low", "close", "adj_close"]
raw_price_columns = ["open", "high", "low", "close"]

# Flag potential issues without modifying any values.
quality_flags = pd.DataFrame(index=deduplicated.index)

quality_flags["missing_price"] = (
    deduplicated[price_columns].isna().any(axis=1)
)
quality_flags["all_raw_prices_missing"] = (
    deduplicated[raw_price_columns].isna().all(axis=1)
)
quality_flags["nonpositive_price"] = (
    deduplicated[price_columns].le(0).any(axis=1)
)
quality_flags["infinite_price"] = (
    np.isinf(deduplicated[price_columns]).any(axis=1)
)

# Compare raw OHLC prices with each other, excluding adjusted close.
quality_flags["invalid_price_range"] = (
    (deduplicated["low"] > deduplicated["high"])
    | (deduplicated["open"] < deduplicated["low"])
    | (deduplicated["open"] > deduplicated["high"])
    | (deduplicated["close"] < deduplicated["low"])
    | (deduplicated["close"] > deduplicated["high"])
)

quality_flags["missing_volume"] = deduplicated["volume"].isna()
quality_flags["zero_volume"] = deduplicated["volume"].eq(0)
quality_flags["invalid_volume"] = (
    deduplicated["volume"].lt(0)
    | np.isinf(deduplicated["volume"])
)

display(quality_flags.sum().rename("affected_rows").to_frame())

,affected_rows
missing_price,18374
all_raw_prices_missing,0
nonpositive_price,5
infinite_price,0
invalid_price_range,5
missing_volume,0
zero_volume,75
invalid_volume,0


In [8]:
# Inspect nonpositive prices and inconsistent raw OHLC ranges.
price_issue_mask = (
    quality_flags["nonpositive_price"]
    | quality_flags["invalid_price_range"]
)

price_issues = deduplicated.loc[
    price_issue_mask,
    ["ticker", "ts", "open", "high", "low", "close", "adj_close", "volume"]
].copy()

display(price_issues)
print("Distinct rows with these price issues:", len(price_issues))

,ticker,ts,open,high,low,close,adj_close,volume
87103,CVX,1962-01-02,0.0,3.30000,3.24000,3.30000,0.335750,105840
167733,JNJ,1962-01-02,0.0,0.22338,0.22222,0.22338,0.064012,0
213412,MCD,1966-07-05,0.0,0.27366,0.26749,0.26955,0.118090,388800
243893,MRK,1962-01-02,0.0,0.39703,0.39317,0.39537,0.068447,633830
287040,PG,1962-01-02,0.0,1.44000,1.42000,1.43000,0.259300,192000


Distinct rows with these price issues: 5


The five zero opening prices are all on the very first date of a ticker's
history. I look them up in the supplied `ohlc` reference table to see
whether it has a usable value.

In [9]:
# Load reference data without modifying the database.
with sqlite3.connect(db_path.as_uri() + "?mode=ro", uri=True) as connection:
    reference = pd.read_sql_query("SELECT * FROM ohlc", connection)

reference["ticker"] = reference["ticker"].map(normalize_ticker)
reference["ts"] = pd.to_datetime(reference["ts"], errors="coerce")

# Display reference observations for the five affected ticker-date pairs.
issue_keys = price_issues[["ticker", "ts"]]

reference_issues = reference.merge(
    issue_keys,
    on=["ticker", "ts"],
    how="inner"
)

display(reference_issues)

# Check whether the affected observations start each ticker's history.
first_dates = deduplicated.groupby("ticker")["ts"].min()
display(first_dates.loc[price_issues["ticker"]].rename("first_date"))

,ts,open,high,low,close,adj_close,volume,ticker
0,1962-01-02,0.0,1.44000,1.42000,1.43000,0.259300,192000,PG
1,1962-01-02,0.0,3.30000,3.24000,3.30000,0.335750,105840,CVX
2,1962-01-02,0.0,0.39703,0.39317,0.39537,0.068447,633830,MRK
3,1962-01-02,0.0,0.22338,0.22222,0.22338,0.064012,0,JNJ
4,1966-07-05,0.0,0.27366,0.26749,0.26955,0.118090,388800,MCD


ticker
CVX   1962-01-02
JNJ   1962-01-02
MCD   1966-07-05
MRK   1962-01-02
PG    1962-01-02
Name: first_date, dtype: datetime64[us]

The reference has the same zero, and there is no previous day to borrow
a price from, so these rows cannot be repaired honestly. They are
quarantined with a reason instead of being deleted silently.

In [10]:
# Quarantine unresolved opening prices at the start of each ticker's history.
quarantine_mask = (
    deduplicated["open"].le(0)
    & deduplicated["ts"].eq(
        deduplicated.groupby("ticker")["ts"].transform("min")
    )
)

quarantined = deduplicated.loc[quarantine_mask].copy()
quarantined["quarantine_reason"] = (
    "Nonpositive opening price at the start of ticker history; "
    "no previous observation and no valid reference opening price."
)

# Keep the remaining observations for further cleaning.
cleaned = deduplicated.loc[~quarantine_mask].copy()

print("Quarantined observations:", len(quarantined))
print("Observations retained for further cleaning:", len(cleaned))

display(quarantined[["ticker", "ts", "open", "quarantine_reason"]])

Quarantined observations: 5
Observations retained for further cleaning: 368246


,ticker,ts,open,quarantine_reason
87103,CVX,1962-01-02,0.0,Nonpositive opening price at the start of tick...
167733,JNJ,1962-01-02,0.0,Nonpositive opening price at the start of tick...
213412,MCD,1966-07-05,0.0,Nonpositive opening price at the start of tick...
243893,MRK,1962-01-02,0.0,Nonpositive opening price at the start of tick...
287040,PG,1962-01-02,0.0,Nonpositive opening price at the start of tick...


### Missing trading days

Missing *cells* and missing *rows* are different problems. For the dates,
I use the supplied `ohlc` table as the reference calendar, over each
ticker's history in that table (plain business days would wrongly count
exchange holidays as gaps). The bounds come from the reference, not from the
data being checked, so a missing first or last day is still detected.

In [11]:
def find_missing_dates(observed, calendar):
    """Return the (ticker, ts) pairs of the calendar absent from observed.

    The calendar defines each ticker's expected dates, including its first
    and last day, so a truncated history is reported as missing dates.
    Only tickers present in observed are checked.
    """
    expected = calendar.loc[
        calendar["ticker"].isin(observed["ticker"]), ["ticker", "ts"]
    ].drop_duplicates()

    comparison = expected.merge(
        observed[["ticker", "ts"]].drop_duplicates(),
        on=["ticker", "ts"],
        how="left",
        indicator=True
    )

    return (
        comparison.loc[comparison["_merge"].eq("left_only"), ["ticker", "ts"]]
        .sort_values(["ticker", "ts"])
        .reset_index(drop=True)
    )


def summarize_missing_dates(calendar, observed, missing):
    """Write one sentence per ticker, including tickers with no gap.

    The period quoted is the ticker's expected history in the calendar.
    """
    span = (
        calendar.loc[calendar["ticker"].isin(observed["ticker"])]
        .groupby("ticker")["ts"].agg(first="min", last="max")
    )
    counts = missing.groupby("ticker").size().reindex(span.index, fill_value=0)

    return [
        f"Ticker {ticker} is missing {counts[ticker]} trading days between "
        f"{span.loc[ticker, 'first'].date()} and {span.loc[ticker, 'last'].date()}"
        for ticker in span.index
    ]


reference_calendar = reference[["ticker", "ts"]].drop_duplicates()
missing_dates = find_missing_dates(deduplicated, reference_calendar)

print("Missing ticker-date observations:", len(missing_dates))
for sentence in summarize_missing_dates(
    reference_calendar, deduplicated, missing_dates
):
    print(sentence)

Missing ticker-date observations: 0
Ticker AAPL is missing 0 trading days between 1980-12-12 and 2024-10-18
Ticker AMGN is missing 0 trading days between 1983-06-17 and 2024-10-18
Ticker AMZN is missing 0 trading days between 1997-05-15 and 2024-12-06
Ticker AXP is missing 0 trading days between 1972-04-03 and 2024-10-18
Ticker BA is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker CAT is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker CRM is missing 0 trading days between 2004-06-23 and 2024-10-18
Ticker CSCO is missing 0 trading days between 1990-02-16 and 2024-10-18
Ticker CVX is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticker DIS is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticker GS is missing 0 trading days between 1999-05-04 and 2024-10-18
Ticker HD is missing 0 trading days between 1981-09-22 and 2024-10-18
Ticker HON is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker IBM is missing 0 trading days bet

In [12]:
# Tiny example: remove one date in the middle of a 5-day history.
toy_calendar = pd.DataFrame({
    "ticker": ["TOY"] * 5,
    "ts": pd.to_datetime([
        "2024-03-04", "2024-03-05", "2024-03-06", "2024-03-07", "2024-03-08"
    ])
})
toy_observed = toy_calendar.drop(index=2)  # 2024-03-06 removed

toy_missing = find_missing_dates(toy_observed, toy_calendar)
assert toy_missing["ts"].tolist() == [pd.Timestamp("2024-03-06")]
print(summarize_missing_dates(toy_calendar, toy_observed, toy_missing)[0])

# Removing the first expected date must also be detected.
toy_missing_first = find_missing_dates(toy_calendar.iloc[1:], toy_calendar)
assert toy_missing_first["ts"].tolist() == [pd.Timestamp("2024-03-04")]

# Nothing removed: zero missing dates.
assert find_missing_dates(toy_calendar, toy_calendar).empty
print("Tests passed: removed middle and first dates are both detected.")

Ticker TOY is missing 1 trading days between 2024-03-04 and 2024-03-08
Tests passed: removed middle and first dates are both detected.


## 1.3 Clean the data

### Missing adjusted close

`adj_close` cannot be forward-filled: a split or dividend between two days
would make the carried value wrong. Instead, when the raw close matches the
reference row for the same (ticker, ts), I take that date's adjustment
factor `adj_close / close` from the reference and apply it to the raw close.

In [13]:
# Match each retained observation with its reference prices.
adjustment_check = cleaned.merge(
    reference[["ticker", "ts", "close", "adj_close"]],
    on=["ticker", "ts"],
    how="left",
    suffixes=("", "_reference"),
    validate="one_to_one"
)

# Check that reference prices are finite and strictly positive.
valid_reference = (
    np.isfinite(adjustment_check["close_reference"])
    & adjustment_check["close_reference"].gt(0)
    & np.isfinite(adjustment_check["adj_close_reference"])
    & adjustment_check["adj_close_reference"].gt(0)
)

# Verify that homework and reference raw closing prices agree.
matching_close = np.isclose(
    adjustment_check["close"],
    adjustment_check["close_reference"],
    rtol=1e-8,
    atol=1e-10
)

missing_adjusted = adjustment_check["adj_close"].isna()
repairable = missing_adjusted & valid_reference & matching_close

print("Missing adjusted closing prices:", missing_adjusted.sum())
print("Missing prices with a validated reference:", repairable.sum())
print("Missing prices requiring further investigation:",
      (missing_adjusted & ~repairable).sum())

display(adjustment_check.loc[
    missing_adjusted,
    ["ticker", "ts", "close", "close_reference", "adj_close_reference"]
].head(10))

Missing adjusted closing prices: 18374
Missing prices with a validated reference: 18374
Missing prices requiring further investigation: 0


,ticker,ts,close,close_reference,adj_close_reference
17,AAPL,1981-01-08,0.135050,0.135050,0.104100
20,AAPL,1981-01-13,0.136160,0.136160,0.104960
42,AAPL,1981-02-12,0.116630,0.116630,0.089907
86,AAPL,1981-04-16,0.111610,0.111610,0.086035
108,AAPL,1981-05-19,0.122770,0.122770,0.094639
119,AAPL,1981-06-04,0.143410,0.143410,0.110560
146,AAPL,1981-07-14,0.106030,0.106030,0.081734
150,AAPL,1981-07-20,0.107700,0.107700,0.083024
176,AAPL,1981-08-26,0.084821,0.084821,0.065387
217,AAPL,1981-10-23,0.084821,0.084821,0.065387


In [14]:
# Preserve the original adjusted prices for auditing.
adjustment_check["adj_close_original"] = adjustment_check["adj_close"]
adjustment_check["adj_close_repaired"] = repairable

# Compute the adjustment factor for each validated observation.
adjustment_factor = (
    adjustment_check.loc[repairable, "adj_close_reference"]
    / adjustment_check.loc[repairable, "close_reference"]
)

# Reconstruct only the missing adjusted closing prices.
adjustment_check.loc[repairable, "adj_close"] = (
    adjustment_check.loc[repairable, "close"] * adjustment_factor
)

# Keep the cleaned data and audit columns.
cleaned = adjustment_check.drop(
    columns=["close_reference", "adj_close_reference"]
).copy()

# Verify that all missing adjusted prices were repaired.
assert cleaned["adj_close"].notna().all()

print("Adjusted closing prices repaired:",
      cleaned["adj_close_repaired"].sum())
print("Remaining missing adjusted closing prices:",
      cleaned["adj_close"].isna().sum())

Adjusted closing prices repaired: 18374
Remaining missing adjusted closing prices: 0


### Extreme returns (3-sigma screen)

Simple returns on the raw close, computed within each ticker after sorting
by date. A return is flagged when it is more than 3 sample standard
deviations (ddof=1) from the ticker's mean, in either direction. This is a
review flag only: flagged rows are not deleted.

In [15]:
def flag_extreme_returns(data, return_column="raw_daily_return", n_std=3):
    """Flag returns more than n_std sample standard deviations from the mean.

    Mean and standard deviation are computed per ticker. Both tails are
    flagged. Undefined returns, and tickers whose standard deviation is zero
    or undefined, are never flagged.
    """
    groups = data.groupby("ticker")[return_column]
    mean_return = groups.transform("mean")
    std_return = groups.transform("std")  # pandas default is ddof=1

    eligible = (
        np.isfinite(data[return_column])
        & np.isfinite(std_return)
        & std_return.gt(0)
    )

    return eligible & (
        (data[return_column] - mean_return).abs().gt(n_std * std_return)
    )


# Small check: both tails flagged, NaN and zero-variance ticker ignored.
toy_returns = pd.DataFrame({
    "ticker": ["A"] * 31 + ["B"] * 3,
    "raw_daily_return": [0.0] * 28 + [0.5, -0.5, np.nan] + [0.01] * 3
})
toy_flags = flag_extreme_returns(toy_returns)
assert toy_flags.tolist() == [False] * 28 + [True, True] + [False] * 4
print("Extreme-return detector test passed.")

# Sort observations before calculating returns within each ticker.
cleaned = cleaned.sort_values(["ticker", "ts"]).reset_index(drop=True)

cleaned["raw_daily_return"] = (
    cleaned.groupby("ticker")["close"].pct_change(fill_method=None)
)
cleaned["extreme_return_flag"] = flag_extreme_returns(cleaned)

print("Extreme daily returns flagged:",
      cleaned["extreme_return_flag"].sum())

display(
    cleaned.loc[
        cleaned["extreme_return_flag"],
        ["ticker", "ts", "close", "raw_daily_return"]
    ].head(20)
)

Extreme-return detector test passed.


Extreme daily returns flagged: 5057


,ticker,ts,close,raw_daily_return
9,AAPL,1980-12-26,0.158480,0.092288
76,AAPL,1981-04-02,0.117750,0.087659
197,AAPL,1981-09-25,0.063616,-0.129776
318,AAPL,1982-03-19,0.074219,0.090173
490,AAPL,1982-11-22,0.125560,-0.089089
495,AAPL,1982-11-30,0.142300,0.103871
531,AAPL,1983-01-20,0.166850,0.111518
593,AAPL,1983-04-20,0.226000,0.088684
660,AAPL,1983-07-26,0.174670,-0.092718
686,AAPL,1983-08-31,0.166290,0.133074


In [16]:
# Match each closing price with the reference.
return_check = cleaned.merge(
    reference[["ticker", "ts", "close"]],
    on=["ticker", "ts"],
    how="left",
    suffixes=("", "_reference"),
    validate="one_to_one"
).sort_values(["ticker", "ts"])

# Compare current closing prices with the reference.
return_check["close_matches_reference"] = np.isclose(
    return_check["close"],
    return_check["close_reference"],
    rtol=1e-8,
    atol=1e-10
)

# Check the previous observation used to calculate each return.
return_check["previous_close_matches_reference"] = (
    return_check.groupby("ticker")["close_matches_reference"]
    .shift(1, fill_value=False)
)

# Inspect flagged returns whose current or previous price differs.
extreme_check = return_check.loc[
    return_check["extreme_return_flag"]
].copy()

price_mismatch = (
    ~extreme_check["close_matches_reference"]
    | ~extreme_check["previous_close_matches_reference"]
)

print("Extreme returns examined:", len(extreme_check))
print("Cases with a current or previous price mismatch:",
      price_mismatch.sum())

display(
    extreme_check.loc[
        price_mismatch,
        ["ticker", "ts", "close", "close_reference", "raw_daily_return"]
    ].head(20)
)

Extreme returns examined: 5057
Cases with a current or previous price mismatch: 0


,ticker,ts,close,close_reference,raw_daily_return


In [17]:
# Inspect the largest absolute returns first.
largest_moves = cleaned.loc[
    cleaned["extreme_return_flag"],
    ["ticker", "ts", "close", "adj_close", "raw_daily_return"]
].copy()

largest_moves["absolute_return"] = largest_moves["raw_daily_return"].abs()

display(
    largest_moves.sort_values(
        "absolute_return", ascending=False
    ).head(20)
)

,ticker,ts,close,adj_close,raw_daily_return,absolute_return
319533,TRV,1981-08-10,12.060000,4.200000,1.000000,1.000000
318327,TRV,1976-10-28,9.220000,3.210000,1.000000,1.000000
318868,TRV,1978-12-20,8.030000,2.800000,0.997512,0.997512
213785,MCD,1968-01-02,0.174900,0.075921,-0.753620,0.753620
4999,AAPL,2000-09-29,0.459820,0.388310,-0.518695,0.518695
319534,TRV,1981-08-11,6.030000,1.570000,-0.500000,0.500000
318328,TRV,1976-10-29,4.690000,0.997070,-0.491323,0.491323
318869,TRV,1978-12-21,4.090000,0.924090,-0.490660,0.490660
280841,NVDA,2000-03-07,0.173570,0.159200,0.424106,0.424106
281950,NVDA,2004-08-06,0.078584,0.072076,-0.352314,0.352314


In [18]:
# Inspect neighboring observations around the three TRV episodes.
trv = cleaned.loc[cleaned["ticker"].eq("TRV")].copy()

# Compare adjusted and raw closing-price movements.
trv["adjusted_daily_return"] = trv["adj_close"].pct_change(
    fill_method=None
)

event_dates = pd.to_datetime([
    "1976-10-28",
    "1978-12-20",
    "1981-08-10"
])

for event_date in event_dates:
    window = trv["ts"].between(
        event_date - pd.Timedelta(days=5),
        event_date + pd.Timedelta(days=5)
    )

    print(f"TRV observations around {event_date.date()}")
    display(
        trv.loc[
            window,
            [
                "ts", "open", "high", "low", "close",
                "adj_close", "volume",
                "raw_daily_return", "adjusted_daily_return"
            ]
        ]
    )

TRV observations around 1976-10-28


,ts,open,high,low,close,adj_close,volume,raw_daily_return,adjusted_daily_return
318324,1976-10-25,4.52,4.52,4.52,4.52,0.96051,29600,0.000000,0.000000
318325,1976-10-26,4.50,4.50,4.50,4.50,0.95719,91200,-0.004425,-0.003456
318326,1976-10-27,4.61,4.61,4.61,4.61,13.88325,230400,0.024444,13.504173
318327,1976-10-28,9.22,9.22,9.22,9.22,3.21000,0,1.000000,-0.768786
318328,1976-10-29,4.69,4.69,4.69,4.69,0.99707,79200,-0.491323,-0.689386
318329,1976-11-01,4.66,4.66,4.66,4.66,0.99043,59200,-0.006397,-0.006660


TRV observations around 1978-12-20


,ts,open,high,low,close,adj_close,volume,raw_daily_return,adjusted_daily_return
318865,1978-12-15,4.19,4.19,4.19,4.19,4.614637,264800,-0.007109,3.845781
318866,1978-12-18,4.02,4.02,4.02,4.02,0.906450,155200,-0.040573,-0.803571
318867,1978-12-19,4.02,4.02,4.02,4.02,0.906450,72000,0.000000,0.000000
318868,1978-12-20,8.03,8.03,8.03,8.03,2.800000,0,0.997512,2.088973
318869,1978-12-21,4.09,4.09,4.09,4.09,0.924090,105600,-0.490660,-0.669968
318870,1978-12-22,4.09,4.09,4.09,4.09,0.936960,91200,0.000000,0.013927


TRV observations around 1981-08-10


,ts,open,high,low,close,adj_close,volume,raw_daily_return,adjusted_daily_return
319530,1981-08-05,6.08,6.08,6.08,6.08,1.58,60800,0.000000,0.000000
319531,1981-08-06,6.06,6.06,6.06,6.06,1.58,60000,-0.003289,0.000000
319532,1981-08-07,6.03,6.03,6.03,6.03,1.57,264800,-0.004950,-0.006329
319533,1981-08-10,12.06,12.06,12.06,12.06,4.20,0,1.000000,1.675159
319534,1981-08-11,6.03,6.03,6.03,6.03,1.57,24800,-0.500000,-0.626190
319535,1981-08-12,6.13,6.13,6.13,6.13,1.59,267200,0.016584,0.012739
319536,1981-08-13,6.03,6.03,6.03,6.03,1.57,155200,-0.016313,-0.012579
319537,1981-08-14,6.00,6.00,6.00,6.00,1.56,145600,-0.004975,-0.006369


In [19]:
# Does the MCD drop reverse the next day (data error) or persist (split-like)?
mcd_window = cleaned.loc[
    cleaned["ticker"].eq("MCD")
    & cleaned["ts"].between("1967-12-20", "1968-01-12"),
    ["ts", "close", "adj_close", "volume"]
]
display(mcd_window)

aapl_window = cleaned.loc[
    cleaned["ticker"].eq("AAPL")
    & cleaned["ts"].between("2000-09-25", "2000-10-04"),
    ["ts", "close", "adj_close", "volume", "raw_daily_return"]
]
display(aapl_window)

,ts,close,adj_close,volume
213778,1967-12-20,0.73354,0.321370,449550
213779,1967-12-21,0.74280,0.325430,400950
213780,1967-12-22,0.74897,0.328130,340200
213781,1967-12-26,0.74074,0.324530,376650
213782,1967-12-27,0.74074,0.324530,498150
213783,1967-12-28,0.73868,0.323630,194400
213784,1967-12-29,0.70988,0.311000,1093500
213785,1968-01-02,0.17490,0.075921,6755143
213786,1968-01-03,0.17567,0.076256,3839207
213787,1968-01-04,0.16204,0.070339,10545887


,ts,close,adj_close,volume,raw_daily_return
4995,2000-09-25,0.95536,0.80677,435551575,0.025152
4996,2000-09-26,0.91853,0.77567,290937144,-0.038551
4997,2000-09-27,0.87388,0.73797,402259544,-0.048610
4998,2000-09-28,0.95536,0.80677,980466003,0.093239
4999,2000-09-29,0.45982,0.38831,3128875900,-0.518695
5000,2000-10-02,0.43304,0.36569,2424791671,-0.058240
5001,2000-10-03,0.39844,0.33647,2038122484,-0.079900
5002,2000-10-04,0.42188,0.35626,1466428216,0.058829


### What the largest flagged moves are

- **TRV 1976-10-28, 1978-12-20, 1981-08-10**: the price doubles for a single
  day with zero volume, then returns to its previous level. The reference
  table has the same prices, so I cannot repair them → quarantined below.
- **MCD 1968-01-02 (−75%)**: the price falls to about 1/4 and *stays* there,
  and volume is roughly ten times higher afterwards. One possible
  explanation is a split that was never adjusted (even `adj_close` drops),
  but the data cannot confirm it and a vendor error is also possible. This
  is only a hypothesis: no correction is applied, the row is kept and flagged.
- **AAPL 2000-09-29 (−52%)**: the drop persists over the following days and
  volume is about three times its usual level, the same in raw and adjusted
  prices. This matches the September 2000 profit warning, but the news itself
  is outside this dataset, so I treat it as a likely real move: kept and flagged.
- The ~5,000 other flags are mostly ordinary volatile days (October 1987,
  early NVDA/AMZN). They remain review flags.

None of these is a confirmed data error that I could interpolate between
valid neighbours, so no return was interpolated.

In [20]:
# Compare adjusted closing prices with the supplied reference.
adjusted_audit = cleaned.merge(
    reference[["ticker", "ts", "adj_close"]],
    on=["ticker", "ts"],
    how="left",
    suffixes=("", "_reference"),
    validate="one_to_one"
)

adjusted_audit["matches_reference"] = np.isclose(
    adjusted_audit["adj_close"],
    adjusted_audit["adj_close_reference"],
    rtol=1e-8,
    atol=1e-10
)

# Inspect discrepancies without changing any prices.
adjusted_discrepancies = adjusted_audit.loc[
    ~adjusted_audit["matches_reference"]
].copy()

print("Adjusted prices differing from the reference:",
      len(adjusted_discrepancies))

display(
    adjusted_discrepancies[
        ["ticker", "ts", "close", "adj_close", "adj_close_reference"]
    ].head(20)
)

Adjusted prices differing from the reference: 10417


,ticker,ts,close,adj_close,adj_close_reference
35,AAPL,1981-02-03,0.123330,1.264587,0.095069
93,AAPL,1981-04-28,0.126120,2.036653,0.097220
101,AAPL,1981-05-08,0.125000,2.129290,0.096360
157,AAPL,1981-07-29,0.106030,0.506329,0.081734
163,AAPL,1981-08-06,0.112720,2.009828,0.086896
167,AAPL,1981-08-13,0.103800,1.122224,0.080013
168,AAPL,1981-08-14,0.102120,1.171347,0.078722
206,AAPL,1981-10-08,0.082589,1.245215,0.063666
264,AAPL,1981-12-31,0.098772,1.730921,0.076141
274,AAPL,1982-01-15,0.089286,1.577125,0.068829


In [21]:
# Validate reference prices for observations with adjusted-price discrepancies.
discrepancy_check = adjusted_discrepancies.merge(
    reference[["ticker", "ts", "close"]].rename(
        columns={"close": "close_reference"}
    ),
    on=["ticker", "ts"],
    how="left",
    validate="one_to_one"
)

matching_raw_close = np.isclose(
    discrepancy_check["close"],
    discrepancy_check["close_reference"],
    rtol=1e-8,
    atol=1e-10
)

valid_reference_prices = (
    np.isfinite(discrepancy_check["close_reference"])
    & discrepancy_check["close_reference"].gt(0)
    & np.isfinite(discrepancy_check["adj_close_reference"])
    & discrepancy_check["adj_close_reference"].gt(0)
)

print("Adjusted-price discrepancies:", len(discrepancy_check))
print("Cases with matching raw closing prices:", matching_raw_close.sum())
print("Cases with valid reference prices:", valid_reference_prices.sum())

# Measure the size of the differences.
discrepancy_check["adjusted_price_ratio"] = (
    discrepancy_check["adj_close"]
    / discrepancy_check["adj_close_reference"]
)

display(
    discrepancy_check[
        ["ticker", "ts", "adj_close",
         "adj_close_reference", "adjusted_price_ratio"]
    ].head(10)
)

Adjusted-price discrepancies: 10417
Cases with matching raw closing prices: 10417
Cases with valid reference prices: 10417


,ticker,ts,adj_close,adj_close_reference,adjusted_price_ratio
0,AAPL,1981-02-03,1.264587,0.095069,13.301729
1,AAPL,1981-04-28,2.036653,0.097220,20.948893
2,AAPL,1981-05-08,2.129290,0.096360,22.097287
3,AAPL,1981-07-29,0.506329,0.081734,6.194843
4,AAPL,1981-08-06,2.009828,0.086896,23.129196
5,AAPL,1981-08-13,1.122224,0.080013,14.025460
6,AAPL,1981-08-14,1.171347,0.078722,14.879513
7,AAPL,1981-10-08,1.245215,0.063666,19.558524
8,AAPL,1981-12-31,1.730921,0.076141,22.733044
9,AAPL,1982-01-15,1.577125,0.068829,22.913784


In [22]:
# Add reference raw closing prices to the adjusted-price audit.
correction = adjusted_audit.merge(
    reference[["ticker", "ts", "close"]].rename(
        columns={"close": "close_reference"}
    ),
    on=["ticker", "ts"],
    how="left",
    validate="one_to_one"
)

# Validate the reference before correcting adjusted-price discrepancies.
valid_source = (
    np.isfinite(correction["close_reference"])
    & correction["close_reference"].gt(0)
    & np.isfinite(correction["adj_close_reference"])
    & correction["adj_close_reference"].gt(0)
    & np.isclose(
        correction["close"],
        correction["close_reference"],
        rtol=1e-8,
        atol=1e-10
    )
)

to_correct = ~correction["matches_reference"] & valid_source

# Record corrections separately from previously filled missing prices.
correction["adj_close_reference_corrected"] = to_correct

# Reconstruct adjusted prices using date-specific reference factors.
factor = (
    correction.loc[to_correct, "adj_close_reference"]
    / correction.loc[to_correct, "close_reference"]
)

correction.loc[to_correct, "adj_close"] = (
    correction.loc[to_correct, "close"] * factor
)

# Preserve original values and repair flags in the cleaned dataset.
cleaned = correction.drop(
    columns=["adj_close_reference", "close_reference", "matches_reference"]
).copy()

assert np.allclose(
    correction.loc[to_correct, "adj_close"],
    correction.loc[to_correct, "adj_close_reference"],
    rtol=1e-8,
    atol=1e-10
)

print("Adjusted-price discrepancies corrected:", to_correct.sum())
print("Discrepancies left unresolved:",
      (~correction["matches_reference"] & ~valid_source).sum())

Adjusted-price discrepancies corrected: 10417
Discrepancies left unresolved: 0


### Impossible price relationships

**Choice: Option 2 (previous day's close).** When `open` or `close` is
outside `[low, high]`, I replace it with the same ticker's previous close,
then check the bounds again. If the previous close does not fit today's
range, if there is no previous day, or if `low > high`, the row is not
repairable and goes to quarantine. When `close` changes, `adj_close` is
rebuilt with that day's own adjustment factor, so the adjusted convention
stays consistent.

The retained data has no such case anymore (the only real ones were the
first-day zero opens, already quarantined), so the function is shown on an
injected example first.

In [23]:
def repair_ohlc_bounds(data):
    """Repair open/close outside [low, high] with the previous close.

    Returns (repaired, unresolved). Rows are unresolved when low > high,
    when there is no usable previous close, or when the replacement still
    violates the day's range.
    """
    result = data.sort_values(["ticker", "ts"]).copy()
    low, high = result["low"], result["high"]

    inverted = low.gt(high)
    bad_open = ~inverted & (result["open"].lt(low) | result["open"].gt(high))
    bad_close = ~inverted & (result["close"].lt(low) | result["close"].gt(high))

    # A previous close that is itself invalid is not reused.
    previous_close = (
        result["close"].mask(bad_close).groupby(result["ticker"]).shift(1)
    )
    factor = result["adj_close"] / result["close"]

    result.loc[bad_open, "open"] = previous_close[bad_open]
    result.loc[bad_close, "close"] = previous_close[bad_close]
    result.loc[bad_close, "adj_close"] = (
        result.loc[bad_close, "close"] * factor[bad_close]
    )
    result["ohlc_repaired"] = bad_open | bad_close

    # Recheck all bounds after the repair.
    still_invalid = (
        inverted
        | result[["open", "close"]].isna().any(axis=1)
        | result["open"].lt(low) | result["open"].gt(high)
        | result["close"].lt(low) | result["close"].gt(high)
    )

    return result.loc[~still_invalid], result.loc[still_invalid]


# Injected example.
toy_prices = pd.DataFrame({
    "ticker": ["T", "T", "T", "T", "U"],
    "ts": pd.to_datetime([
        "2024-01-02", "2024-01-03", "2024-01-04", "2024-01-05", "2024-01-02"
    ]),
    "open":      [10.0, 10.2, 10.1, 10.3, 0.0],
    "high":      [10.5, 10.6, 10.4, 10.4, 5.0],
    "low":       [ 9.8, 10.0,  9.9, 10.5, 4.5],
    "close":     [10.2, 11.0, 10.0, 10.4, 4.8],
    "adj_close": [ 5.1,  5.5,  5.0,  5.2, 2.4],
})
# Row 1: close 11.0 > high 10.6  -> repairable with previous close 10.2
# Row 3: low 10.5 > high 10.4    -> unresolved
# Row 4: open 0 on the first day -> unresolved (no previous close)

toy_repaired, toy_unresolved = repair_ohlc_bounds(toy_prices)

assert toy_repaired.loc[1, "close"] == 10.2
assert np.isclose(toy_repaired.loc[1, "adj_close"], 10.2 * 5.5 / 11.0)
assert toy_repaired.loc[1, "ohlc_repaired"]
assert sorted(toy_unresolved.index) == [3, 4]
inside = toy_repaired[["open", "close"]]
assert (
    inside.min(axis=1).ge(toy_repaired["low"])
    & inside.max(axis=1).le(toy_repaired["high"])
).all()

print("OHLC repair test passed.")
display(toy_repaired)
display(toy_unresolved)

OHLC repair test passed.


,ticker,ts,open,high,low,close,adj_close,ohlc_repaired
0,T,2024-01-02,10.0,10.5,9.8,10.2,5.1,False
1,T,2024-01-03,10.2,10.6,10.0,10.2,5.1,True
2,T,2024-01-04,10.1,10.4,9.9,10.0,5.0,False


,ticker,ts,open,high,low,close,adj_close,ohlc_repaired
3,T,2024-01-05,10.3,10.4,10.5,10.4,5.2,False
4,U,2024-01-02,NaN,5.0,4.5,4.8,2.4,True


In [24]:
# Apply the same function to the retained observations.
cleaned, ohlc_unresolved = repair_ohlc_bounds(cleaned)
cleaned = cleaned.reset_index(drop=True)

ohlc_unresolved = ohlc_unresolved.assign(
    quarantine_reason="OHLC bounds could not be repaired with the previous close."
)
quarantined = pd.concat([quarantined, ohlc_unresolved], ignore_index=True)

print("Rows repaired:", cleaned["ohlc_repaired"].sum())
print("Rows sent to quarantine:", len(ohlc_unresolved))

Rows repaired: 0
Rows sent to quarantine: 0


In [25]:
print("Dataset shape:", cleaned.shape)

print("\nColumns:")
print(cleaned.columns.tolist())

print("\nFirst five rows:")
display(cleaned.head())

Dataset shape: (368246, 15)

Columns:
['ts', 'open', 'high', 'low', 'close', 'adj_close', 'volume', 'ticker', 'ticker_original', 'adj_close_original', 'adj_close_repaired', 'raw_daily_return', 'extreme_return_flag', 'adj_close_reference_corrected', 'ohlc_repaired']

First five rows:


,ts,open,high,low,close,adj_close,volume,ticker,ticker_original,adj_close_original,adj_close_repaired,raw_daily_return,extreme_return_flag,adj_close_reference_corrected,ohlc_repaired
0,1980-12-12,0.12835,0.12835,0.12835,0.12835,0.098941,469034068,AAPL,AAPL,0.098941,False,NaN,False,False,False
1,1980-12-15,0.12165,0.12165,0.12165,0.12165,0.093779,175884971,AAPL,AAPL,0.093779,False,-0.052201,False,False,False
2,1980-12-16,0.11272,0.11272,0.11272,0.11272,0.086896,105728103,AAPL,AAPL,0.086896,False,-0.073407,False,False,False
3,1980-12-17,0.11551,0.11551,0.11551,0.11551,0.089046,86441684,AAPL,AAPL,0.089046,False,0.024752,False,False,False
4,1980-12-18,0.11886,0.11886,0.11886,0.11886,0.091627,73449671,AAPL,AAPL,0.091627,False,0.029002,False,False,False


In [26]:
# Check missing values in the cleaned dataset.
missing_summary = pd.DataFrame({
    "Missing Values": cleaned.isna().sum(),
    "Missing Percentage (%)": cleaned.isna().mean() * 100
})

display(missing_summary)

,Missing Values,Missing Percentage (%)
ts,0,0.000000
open,0,0.000000
high,0,0.000000
low,0,0.000000
close,0,0.000000
adj_close,0,0.000000
volume,0,0.000000
ticker,0,0.000000
ticker_original,0,0.000000
adj_close_original,18374,4.989599


In [27]:
# Identify observations with undefined daily returns.
missing_returns = cleaned.loc[
    cleaned["raw_daily_return"].isna()
].copy()

# Identify the first available date for each ticker.
first_dates = cleaned.groupby("ticker")["ts"].min()

missing_returns["is_first_observation"] = (
    missing_returns["ts"]
    == missing_returns["ticker"].map(first_dates)
)

print("Missing daily returns:", len(missing_returns))
print("Affected tickers:", missing_returns["ticker"].nunique())
print("Missing returns on first observations:",
      missing_returns["is_first_observation"].sum())
print("Missing returns elsewhere:",
      (~missing_returns["is_first_observation"]).sum())

# Verify exactly one undefined initial return per ticker.
assert missing_returns["is_first_observation"].all()
assert missing_returns["ticker"].is_unique
assert len(missing_returns) == cleaned["ticker"].nunique()

print("Validation passed: one undefined initial return per ticker.")

Missing daily returns: 30
Affected tickers: 30
Missing returns on first observations: 30
Missing returns elsewhere: 0


Validation passed: one undefined initial return per ticker.


### Quarantining Three Unresolved TRV Observations

Inspection identified three TRV observations with approximately doubled
raw prices, zero reported volume, and a reversal toward the previous
price level on the following session:

- 1976-10-28
- 1978-12-20
- 1981-08-10

The same raw closing prices occur in the supplied reference, so that
reference cannot resolve these cases. The observed pattern is suspicious,
but it does not establish a data error or a specific corporate action.

We quarantine these three observations rather than invent replacement
prices. Their values and the reason for exclusion are preserved.
Other extreme returns and zero-volume observations are not automatically
removed.

Returns must be recalculated after this exclusion. A return spanning
an excluded session must be distinguished from a one-session return.

In [28]:
# Preserve the dataset before excluding unresolved TRV observations.
cleaned_before_trv = cleaned.copy()

trv_suspect_dates = pd.to_datetime([
    "1976-10-28",
    "1978-12-20",
    "1981-08-10"
])

trv_quarantine_mask = (
    cleaned["ticker"].eq("TRV")
    & cleaned["ts"].isin(trv_suspect_dates)
)

trv_quarantined = cleaned.loc[trv_quarantine_mask].copy()

trv_quarantined["quarantine_reason"] = (
    "Unresolved TRV price spike with zero reported volume and "
    "next-session reversal; reference shares the same raw prices."
)

# Preserve excluded observations alongside the earlier quarantine.
quarantined = pd.concat(
    [quarantined, trv_quarantined],
    ignore_index=True
).drop_duplicates(subset=["ticker", "ts"])

cleaned = cleaned.loc[~trv_quarantine_mask].copy()

print("TRV observations quarantined:", len(trv_quarantined))
print("Total quarantined observations:", len(quarantined))
print("Retained observations:", len(cleaned))

display(
    trv_quarantined[
        ["ticker", "ts", "close", "volume", "quarantine_reason"]
    ]
)

TRV observations quarantined: 3
Total quarantined observations: 8
Retained observations: 368243


,ticker,ts,close,volume,quarantine_reason
318327,TRV,1976-10-28,9.22,0,Unresolved TRV price spike with zero reported ...
318868,TRV,1978-12-20,8.03,0,Unresolved TRV price spike with zero reported ...
319533,TRV,1981-08-10,12.06,0,Unresolved TRV price spike with zero reported ...


### Recalculating Returns After Quarantine

Removing an observation changes which prices are adjacent in the
retained dataset. A price change spanning an excluded trading session
must not be treated as a one-session return.

We use each ticker's supplied reference dates to identify its expected
previous session. Returns are left undefined when the previous retained
observation is not that expected session.

We preserve the original extreme-return flags for auditing, then
recalculate the three-standard-deviation screen using eligible
one-session raw closing-price returns.

In [29]:
# Build the expected previous-session calendar for each ticker.
calendar = (
    reference[["ticker", "ts"]]
    .drop_duplicates()
    .sort_values(["ticker", "ts"])
)

calendar["expected_previous_date"] = (
    calendar.groupby("ticker")["ts"].shift(1)
)

# Preserve the initial diagnostic flags.
cleaned["extreme_return_flag_before_quarantine"] = (
    cleaned["extreme_return_flag"]
)

cleaned = cleaned.merge(
    calendar,
    on=["ticker", "ts"],
    how="left",
    validate="one_to_one"
).sort_values(["ticker", "ts"]).reset_index(drop=True)

previous_retained_date = cleaned.groupby("ticker")["ts"].shift(1)

# Flag returns that would span an excluded reference session.
cleaned["return_gap_flag"] = (
    previous_retained_date.notna()
    & previous_retained_date.ne(cleaned["expected_previous_date"])
)

# Calculate returns only between consecutive reference sessions.
cleaned["raw_daily_return"] = (
    cleaned.groupby("ticker")["close"]
    .pct_change(fill_method=None)
    .mask(cleaned["return_gap_flag"])
)

# Recalculate the retrospective extreme-return screen.
cleaned["extreme_return_flag"] = flag_extreme_returns(cleaned)

print("Returns spanning excluded sessions:",
      cleaned["return_gap_flag"].sum())
print("Undefined daily returns:",
      cleaned["raw_daily_return"].isna().sum())
print("Extreme returns after recalculation:",
      cleaned["extreme_return_flag"].sum())

display(
    cleaned.loc[
        cleaned["return_gap_flag"],
        ["ticker", "ts", "expected_previous_date", "raw_daily_return"]
    ]
)

Returns spanning excluded sessions: 3
Undefined daily returns: 33
Extreme returns after recalculation: 5176


,ticker,ts,expected_previous_date,raw_daily_return
318327,TRV,1976-10-29,1976-10-28,NaN
318867,TRV,1978-12-21,1978-12-20,NaN
319531,TRV,1981-08-11,1981-08-10,NaN


### Structural Validation After Repairs and Quarantine

We validate the retained observations using a reusable function.
All raw and adjusted prices must be finite and strictly positive.
Raw OHLC prices must satisfy low <= open <= high and
low <= close <= high.

Volumes must be finite and nonnegative, dates and ticker labels must
be present, and ticker-date keys must be unique.

Zero volume and extreme returns are review flags rather than automatic
validation failures. Audit columns and return columns may contain
documented missing values.

In [30]:
def validate_market_data(data):
    """Count observations violating required market-data constraints."""
    prices = data[["open", "high", "low", "close", "adj_close"]]

    invalid_bounds = (
        data["low"].gt(data["high"])
        | data["open"].lt(data["low"])
        | data["open"].gt(data["high"])
        | data["close"].lt(data["low"])
        | data["close"].gt(data["high"])
    )

    checks = {
        "missing_price": prices.isna().any(axis=1),
        "nonfinite_price": (~np.isfinite(prices)).any(axis=1),
        "nonpositive_price": prices.le(0).any(axis=1),
        "invalid_ohlc_bounds": invalid_bounds,
        "invalid_volume": (
            ~np.isfinite(data["volume"])
            | data["volume"].lt(0)
        ),
        "missing_date": data["ts"].isna(),
        "invalid_ticker": (
            data["ticker"].isna()
            | data["ticker"].fillna("").str.strip().eq("")
        ),
        "duplicate_ticker_date": data.duplicated(
            subset=["ticker", "ts"], keep=False
        )
    }

    return pd.Series({
        name: int(mask.sum())
        for name, mask in checks.items()
    }, name="affected_rows")


validation_summary = validate_market_data(cleaned)
display(validation_summary.to_frame())

assert validation_summary.eq(0).all(), (
    "Validation failed: inspect the reported violations."
)

print("Validation passed: all retained observations satisfy the constraints.")

,affected_rows
missing_price,0
nonfinite_price,0
nonpositive_price,0
invalid_ohlc_bounds,0
invalid_volume,0
missing_date,0
invalid_ticker,0
duplicate_ticker_date,0


Validation passed: all retained observations satisfy the constraints.


### Testing the Validator with Injected Errors

Some required error cases are absent from the retained dataset.
We therefore inject individual errors into small independent test copies.

The tests cover missing, infinite and nonpositive prices, invalid OHLC
bounds, missing volume, missing dates, empty tickers and duplicate keys.
Each test must trigger its corresponding validation check.

These tests do not modify the working dataset. Repair behavior is
tested separately.

In [31]:
# Use a small valid sample as the starting point.
test_base = cleaned.head(3).copy().reset_index(drop=True)
assert validate_market_data(test_base).eq(0).all()

# Define individual errors and the checks they must trigger.
test_cases = [
    ("missing_price", "adj_close", np.nan),
    ("nonfinite_price", "adj_close", np.inf),
    ("nonpositive_price", "adj_close", -1.0),
    ("invalid_ohlc_bounds", "high", 0.0),
    ("invalid_volume", "volume", np.nan),
    ("missing_date", "ts", pd.NaT),
    ("invalid_ticker", "ticker", "")
]

for expected_issue, column, value in test_cases:
    test_data = test_base.copy()

    # Allow missing volume even if the source column contains integers.
    if column == "volume":
        test_data["volume"] = test_data["volume"].astype(float)

    test_data.loc[0, column] = value
    result = validate_market_data(test_data)

    assert result[expected_issue] == 1, (
        f"Detector failed for: {expected_issue}"
    )
    print(f"Test passed: {expected_issue}")

# Verify that both observations sharing a key are detected.
duplicate_test = pd.concat(
    [test_base, test_base.iloc[[0]]],
    ignore_index=True
)

assert validate_market_data(duplicate_test)["duplicate_ticker_date"] == 2
print("Test passed: duplicate_ticker_date")

# Confirm that the original test sample remains valid.
assert validate_market_data(test_base).eq(0).all()
print("All injected-error validation tests passed.")

Test passed: missing_price
Test passed: nonfinite_price
Test passed: nonpositive_price
Test passed: invalid_ohlc_bounds
Test passed: invalid_volume
Test passed: missing_date
Test passed: invalid_ticker
Test passed: duplicate_ticker_date
All injected-error validation tests passed.


### Data Quality Report

This report preserves issues detected before repairs, together with
documented quarantine decisions. Multiple issues may refer to the same
observation; issue counts must therefore not be added to estimate the
number of affected observations.

Severity levels are assigned as follows:

- **High:** invalid or missing prices, inconsistent OHLC bounds, invalid
  volume, adjusted-price discrepancies, or unresolved quarantined prices.
- **Medium:** exact duplicate records, missing reference dates, zero
  reported volume, or extreme returns requiring investigation.
- **Low:** ticker formatting corrected during normalization.

Extreme-return flags describe a statistical screen, not confirmed errors.
Zero reported volume does not establish that no trading occurred.

In [32]:
def make_issue_rows(data, issue_type, description, severity):
    """Convert observations into rows with the required report schema."""
    rows = data[["ticker", "ts"]].copy()
    rows = rows.rename(columns={"ts": "issue_date"})
    rows["issue_type"] = issue_type
    rows["description"] = description
    rows["severity"] = severity

    return rows[
        ["ticker", "issue_type", "issue_date", "description", "severity"]
    ]


# Preserve the price and volume diagnosis made before repairs.
issue_definitions = {
    "missing_price": (
        "At least one price was missing before repair.", "high"
    ),
    "all_raw_prices_missing": (
        "All four raw OHLC prices were missing.", "high"
    ),
    "nonpositive_price": (
        "At least one price was zero or negative.", "high"
    ),
    "infinite_price": (
        "At least one price was infinite.", "high"
    ),
    "invalid_price_range": (
        "Raw OHLC prices violated their required bounds.", "high"
    ),
    "missing_volume": (
        "Reported volume was missing.", "high"
    ),
    "zero_volume": (
        "Reported volume was zero; trading activity is uncertain.", "medium"
    ),
    "invalid_volume": (
        "Reported volume was negative or infinite.", "high"
    )
}

report_parts = []

for issue_type, (description, severity) in issue_definitions.items():
    affected = deduplicated.loc[quality_flags[issue_type]]
    report_parts.append(
        make_issue_rows(affected, issue_type, description, severity)
    )

# Add duplicate records and normalized ticker labels.
report_parts.append(make_issue_rows(
    exact_duplicates, "exact_duplicate",
    "Repeated normalized record removed; one copy retained.", "medium"
))

changed_labels = working.loc[
    working["ticker_original"].ne(working["ticker"])
]

report_parts.append(make_issue_rows(
    changed_labels, "ticker_format",
    "Original ticker label normalized; original label preserved.", "low"
))

# Add adjusted-price discrepancies detected before correction.
report_parts.append(make_issue_rows(
    adjusted_discrepancies, "adjusted_price_discrepancy",
    "Adjusted price differed from the supplied reference convention.",
    "high"
))

# Preserve the extreme-return screen performed before TRV quarantine.
initial_extremes = cleaned_before_trv.loc[
    cleaned_before_trv["extreme_return_flag"]
]

report_parts.append(make_issue_rows(
    initial_extremes, "extreme_raw_return",
    "Pre-TRV-quarantine raw return exceeded the ticker's three-SD threshold; "
    "review flag, not a confirmed error.",
    "medium"
))

# Include absent reference dates, even when this table is empty.
report_parts.append(make_issue_rows(
    missing_dates, "missing_reference_date",
    "Ticker-date observation absent before quarantine relative to reference.",
    "medium"
))

# Record the reasons for all quarantine decisions.
quarantine_report = make_issue_rows(
    quarantined, "quarantined", "", "high"
)
quarantine_report["description"] = quarantined[
    "quarantine_reason"
].to_numpy()
report_parts.append(quarantine_report)

data_quality_report = pd.concat(report_parts, ignore_index=True)
data_quality_report = data_quality_report.sort_values(
    ["ticker", "issue_date", "issue_type"]
).reset_index(drop=True)

display(
    data_quality_report.groupby(
        ["issue_type", "severity"]
    ).size().rename("issue_count").to_frame()
)

display(data_quality_report.head(10))

,,issue_count
issue_type,severity,
adjusted_price_discrepancy,high,10417
exact_duplicate,medium,29460
extreme_raw_return,medium,5057
invalid_price_range,high,5
missing_price,high,18374
nonpositive_price,high,5
quarantined,high,8
ticker_format,low,39936
zero_volume,medium,75


,ticker,issue_type,issue_date,description,severity
0,AAPL,ticker_format,1980-12-19,Original ticker label normalized; original lab...,low
1,AAPL,extreme_raw_return,1980-12-26,Pre-TRV-quarantine raw return exceeded the tic...,medium
2,AAPL,ticker_format,1980-12-29,Original ticker label normalized; original lab...,low
3,AAPL,exact_duplicate,1981-01-02,Repeated normalized record removed; one copy r...,medium
4,AAPL,exact_duplicate,1981-01-05,Repeated normalized record removed; one copy r...,medium
5,AAPL,missing_price,1981-01-08,At least one price was missing before repair.,high
6,AAPL,missing_price,1981-01-13,At least one price was missing before repair.,high
7,AAPL,ticker_format,1981-02-02,Original ticker label normalized; original lab...,low
8,AAPL,adjusted_price_discrepancy,1981-02-03,Adjusted price differed from the supplied refe...,high
9,AAPL,ticker_format,1981-02-04,Original ticker label normalized; original lab...,low


### Handling Missing Raw Prices and Volume

The function below first removes observations where all four raw OHLC
prices are missing. It then forward-fills remaining missing raw prices
within each ticker, in date order, and replaces missing volume with zero.

Repair flags distinguish imputed values from original observations.
A filled volume of zero represents unknown activity, not confirmed
absence of trading. Leading price gaps remain unresolved.

This function does not reconstruct adjusted prices. Any repaired raw
closing price requires a separate adjustment-consistency check.
All repaired observations must also pass OHLC validation; forward-fill
alone does not guarantee valid bounds.

In [33]:
def fill_missing_raw_values(data):
    """Fill raw-price gaps within tickers and flag volume imputations."""
    raw_columns = ["open", "high", "low", "close"]
    result = data.sort_values(["ticker", "ts"]).copy()

    # Remove fully missing raw-price observations before filling.
    all_missing = result[raw_columns].isna().all(axis=1)
    dropped = result.loc[all_missing].copy()
    result = result.loc[~all_missing].copy()

    missing_before = result[raw_columns].isna()

    result[raw_columns] = (
        result.groupby("ticker")[raw_columns].ffill()
    )

    # Record actual fills, including whether close was reconstructed.
    filled = missing_before & result[raw_columns].notna()
    result["raw_price_filled"] = filled.any(axis=1)
    result["raw_close_filled"] = filled["close"]

    result["volume_imputed"] = result["volume"].isna()
    result["volume"] = result["volume"].fillna(0)

    return result, dropped


# Demonstrate the behavior using independent synthetic observations.
example = pd.DataFrame({
    "ticker": ["TEST_A", "TEST_A", "TEST_A", "TEST_B"],
    "ts": pd.to_datetime([
        "2024-01-02", "2024-01-03", "2024-01-04", "2024-01-02"
    ]),
    "open": [10.0, np.nan, np.nan, np.nan],
    "high": [12.0, 12.0, np.nan, 22.0],
    "low": [9.0, 9.0, np.nan, 19.0],
    "close": [11.0, 11.0, np.nan, 21.0],
    "volume": [100.0, np.nan, 100.0, 200.0]
})

example_filled, example_dropped = fill_missing_raw_values(example)

# An internal missing opening price uses the same ticker's prior value.
assert example_filled.loc[1, "open"] == 10.0
assert example_filled.loc[1, "raw_price_filled"]

# Missing volume becomes zero and remains explicitly flagged.
assert example_filled.loc[1, "volume"] == 0
assert example_filled.loc[1, "volume_imputed"]

# Fully missing OHLC rows are removed before filling.
assert example_dropped.index.tolist() == [2]

# A leading gap must not borrow another ticker's price.
assert pd.isna(example_filled.loc[3, "open"])

print("Missing-value repair tests passed.")
display(example_filled)

Missing-value repair tests passed.


,ticker,ts,open,high,low,close,volume,raw_price_filled,raw_close_filled,volume_imputed
0,TEST_A,2024-01-02,10.0,12.0,9.0,11.0,100.0,False,False,False
1,TEST_A,2024-01-03,10.0,12.0,9.0,11.0,0.0,True,False,True
3,TEST_B,2024-01-02,NaN,22.0,19.0,21.0,200.0,False,False,False


### Final Consistency Checks and Row Reconciliation

We apply the tested missing-value function to the retained dataset.
No additional fills are expected because its raw prices and volumes
are already complete.

We then recheck structural constraints and agreement with the adopted
reference adjustment convention. Reference agreement confirms consistency
with that convention, not independent historical accuracy.

Finally, we reconcile original observations with retained rows,
removed duplicates, fully missing raw-price rows and quarantined rows,
and display descriptive statistics before and after cleaning.

In [34]:
# Apply the tested function to the actual retained observations.
cleaned, dropped_all_missing = fill_missing_raw_values(cleaned)

print("Additional raw-price fills:", cleaned["raw_price_filled"].sum())
print("Additional volume imputations:", cleaned["volume_imputed"].sum())
print("Fully missing raw-price rows removed:", len(dropped_all_missing))

# Zero volume is kept but stored as an explicit review flag.
cleaned["zero_volume_flag"] = cleaned["volume"].eq(0)
print("Zero-volume rows flagged:", cleaned["zero_volume_flag"].sum())

# Any new price repair would require investigation before proceeding.
assert not cleaned["raw_price_filled"].any()
assert not cleaned["volume_imputed"].any()
assert validate_market_data(cleaned).eq(0).all()

# Recheck the adopted adjusted-price convention.
final_adjustment_check = cleaned.merge(
    reference[["ticker", "ts", "adj_close"]],
    on=["ticker", "ts"],
    how="left",
    suffixes=("", "_reference"),
    validate="one_to_one"
)

assert np.allclose(
    final_adjustment_check["adj_close"],
    final_adjustment_check["adj_close_reference"],
    rtol=1e-8,
    atol=1e-10
)

# Reconcile every original row with its final disposition.
row_accounting = pd.Series({
    "original_rows": len(df),
    "duplicates_removed": len(exact_duplicates),
    "fully_missing_raw_rows_removed": len(dropped_all_missing),
    "quarantined_rows": len(quarantined),
    "retained_rows": len(cleaned)
}, name="row_count")

assert len(df) == (
    len(exact_duplicates)
    + len(dropped_all_missing)
    + len(quarantined)
    + len(cleaned)
)

display(row_accounting.to_frame())

# Compare the original and retained price/volume distributions.
comparison_columns = ["open", "high", "low", "close", "adj_close", "volume"]

display(pd.concat({
    "Before cleaning": df[comparison_columns].describe(),
    "After cleaning": cleaned[comparison_columns].describe()
}, axis=1))

print("Consistency checks and row reconciliation passed.")

Additional raw-price fills: 0
Additional volume imputations: 0
Fully missing raw-price rows removed: 0
Zero-volume rows flagged: 71


,row_count
original_rows,397711
duplicates_removed,29460
fully_missing_raw_rows_removed,0
quarantined_rows,8
retained_rows,368243


Before cleaning                                               \
                 open           high            low          close   
count   397711.000000  397711.000000  397711.000000  397711.000000   
mean        46.285479      46.747660      45.816279      46.292903   
std         67.132594      67.778210      66.477178      67.141879   
min          0.000000       0.005208       0.004801       0.005208   
25%          4.209938       4.240000       4.160000       4.200000   
50%         19.850000      20.060000      19.630000      19.860000   
75%         57.530000      58.130000      56.950000      57.530000   
max        604.260000     608.630000     597.880000     605.400000   

                                   After cleaning                 \
           adj_close        volume           open           high   
count  377856.000000  3.977110e+05  368243.000000  368243.000000   
mean       51.652250  3.097773e+07      46.309692      46.772125   
std       194.584045  1.266237e+08      67.143446      67.789513   
min         0.002945  0.000000e+00       0.005208       0.005208   
25%         1.390000  1.730700e+06       4.210000       4.250000   
50%        11.960000  4.648200e+06      19.890000      20.090000   
75%        43.770000  1.102380e+07      57.565000      58.160000   
max     10442.510783  9.276606e+09     604.260000     608.630000   

                                                                  
                 low          close      adj_close        volume  
count  368243.000000  368243.000000  368243.000000  3.682430e+05  
mean       45.840669      46.317623      37.274946  3.095209e+07  
std        66.489641      67.155132      63.588058  1.266158e+08  
min         0.004801       0.005208       0.002945  0.000000e+00  
25%         4.170000       4.210000       1.310000  1.731650e+06  
50%        19.670000      19.910000      11.460000  4.649780e+06  
75%        57.000000      57.570000      41.570000  1.102171e+07  
max       597.880000     605.400000     605.400000  9.276606e+09

Consistency checks and row reconciliation passed.


### Missing dates after quarantine

The "zero missing days" result above was computed *before* quarantine.
The 8 quarantined observations are now intentionally absent from the cleaned
data, so the same detector must find exactly those 8 (ticker, ts) keys and
nothing else. I also check the other direction: every `ohlc_hw` key must exist
in the reference calendar.

In [35]:
# Key-by-key comparison, not just first/last dates and counts.
missing_after_cleaning = find_missing_dates(cleaned, reference_calendar)

quarantined_keys = set(zip(quarantined["ticker"], quarantined["ts"]))
missing_keys = set(zip(missing_after_cleaning["ticker"], missing_after_cleaning["ts"]))
assert missing_keys == quarantined_keys

# Other direction, before quarantine (so it also covers the cleaned rows):
# no ohlc_hw (ticker, ts) key outside the reference calendar.
extra_keys = deduplicated[["ticker", "ts"]].merge(
    reference_calendar, on=["ticker", "ts"], how="left", indicator=True
)
assert extra_keys["_merge"].eq("both").all()

print("Dates absent after cleaning:", len(missing_after_cleaning),
      "(all of them quarantined on purpose)")
display(
    missing_after_cleaning.merge(
        quarantined[["ticker", "ts", "quarantine_reason"]],
        on=["ticker", "ts"]
    )
)
print("ohlc_hw dates not in the reference calendar:",
      (~extra_keys["_merge"].eq("both")).sum())

Dates absent after cleaning: 8 (all of them quarantined on purpose)


,ticker,ts,quarantine_reason
0,CVX,1962-01-02,Nonpositive opening price at the start of tick...
1,JNJ,1962-01-02,Nonpositive opening price at the start of tick...
2,MCD,1966-07-05,Nonpositive opening price at the start of tick...
3,MRK,1962-01-02,Nonpositive opening price at the start of tick...
4,PG,1962-01-02,Nonpositive opening price at the start of tick...
5,TRV,1976-10-28,Unresolved TRV price spike with zero reported ...
6,TRV,1978-12-20,Unresolved TRV price spike with zero reported ...
7,TRV,1981-08-10,Unresolved TRV price spike with zero reported ...


ohlc_hw dates not in the reference calendar: 0


### Saving the Cleaned Data and Audit Tables

The validated dataset is saved as `ohlc_hw_cleaned` in the local working
database. Repair flags and original-value audit columns are retained.

Quarantined observations and the data quality report are saved separately.
Dates are serialized as ISO dates (YYYY-MM-DD) for reliable SQL filtering.

Only generated output tables are replaced when this cell is rerun.
The source `ohlc_hw` table remains unchanged, which is verified by
comparing its contents before and after saving.

In [36]:
def prepare_for_sql(data):
    """Serialize date columns as ISO dates without modifying the input."""
    output = data.copy()

    for column in ["ts", "issue_date", "expected_previous_date"]:
        if column in output.columns:
            output[column] = pd.to_datetime(
                output[column], errors="raise"
            ).dt.strftime("%Y-%m-%d")

    return output


# Save generated tables in the local homework database.
with sqlite3.connect(db_path) as connection:
    raw_before_save = pd.read_sql_query(
        "SELECT * FROM ohlc_hw", connection
    )

    pd.testing.assert_frame_equal(raw_before_save, df)

    prepare_for_sql(cleaned).to_sql(
        "ohlc_hw_cleaned",
        connection,
        if_exists="replace",
        index=False
    )

    prepare_for_sql(quarantined).to_sql(
        "ohlc_hw_quarantined",
        connection,
        if_exists="replace",
        index=False
    )

    prepare_for_sql(data_quality_report).to_sql(
        "ohlc_hw_quality_report",
        connection,
        if_exists="replace",
        index=False
    )

    # Read the saved data back for verification.
    saved_cleaned = pd.read_sql_query(
        "SELECT * FROM ohlc_hw_cleaned",
        connection,
        parse_dates=["ts"]
    )

    raw_after_save = pd.read_sql_query(
        "SELECT * FROM ohlc_hw", connection
    )

# Verify the saved observations and preservation of the source table.
pd.testing.assert_frame_equal(raw_before_save, raw_after_save)
assert len(saved_cleaned) == len(cleaned)
assert validate_market_data(saved_cleaned).eq(0).all()

# Check that prices survived the database round trip.
price_columns = ["open", "high", "low", "close", "adj_close"]
expected_prices = cleaned.sort_values(["ticker", "ts"])[price_columns]
saved_prices = saved_cleaned.sort_values(["ticker", "ts"])[price_columns]

np.testing.assert_allclose(
    saved_prices.to_numpy(),
    expected_prices.to_numpy(),
    rtol=1e-12,
    atol=1e-12
)

print("Saved cleaned observations:", len(saved_cleaned))
print("Saved quarantined observations:", len(quarantined))
print("Saved quality-report entries:", len(data_quality_report))
print("Database verification passed. Source ohlc_hw is unchanged.")

Saved cleaned observations: 368243
Saved quarantined observations: 8
Saved quality-report entries: 103337
Database verification passed. Source ohlc_hw is unchanged.
